In [1]:
# ============================================================================
# MSGR-RAG Component 6: End-to-End RAG Pipeline (Question -> Verified Answer)
# ============================================================================
# Purpose: Close the loop - take a free-text user question, extract candidate
# facts, retrieve supporting context, generate an answer, then verify every
# claim in that answer using the validated Component 5 multi-scale verifier.
#
# This is a minimal, transparent proof-of-concept (not a production system),
# built on top of the already-validated verification core. It reuses G_E, G_C,
# G_D and the entity_to_concept / concept_to_domain / entity_to_domain_atc
# mappings exactly as Component 5 produced them - nothing about the verified
# core is modified here.
# ============================================================================

import pandas as pd
import numpy as np
import pickle
import re
from pathlib import Path

INTERMEDIATE_DIR = Path("../Datasets/intermediate")

# ---- Load the three validated knowledge graphs ----
with open(INTERMEDIATE_DIR / "graph_entity.gpickle", "rb") as f:
    G_E = pickle.load(f)
with open(INTERMEDIATE_DIR / "graph_concept.gpickle", "rb") as f:
    G_C = pickle.load(f)
with open(INTERMEDIATE_DIR / "graph_domain.gpickle", "rb") as f:
    G_D = pickle.load(f)

print(f"Loaded G_E: {G_E.number_of_nodes()} nodes, {G_E.number_of_edges()} edges")
print(f"Loaded G_C: {G_C.number_of_nodes()} nodes, {G_C.number_of_edges()} edges")
print(f"Loaded G_D: {G_D.number_of_nodes()} nodes, {G_D.number_of_edges()} edges")

# ---- Load abstraction mappings (phi_EC, phi_CD, ATC-grounded phi_ED override) ----
concept_mappings_df = pd.read_csv(INTERMEDIATE_DIR / "concept_mappings.csv")
domain_mappings_df = pd.read_csv(INTERMEDIATE_DIR / "domain_mappings.csv")
atc_overrides_df = pd.read_csv(INTERMEDIATE_DIR / "atc_entity_domain_overrides.csv")

entity_to_concept = dict(zip(concept_mappings_df['entity_name'], concept_mappings_df['concept_name']))
concept_to_domain = dict(zip(domain_mappings_df['concept_name'], domain_mappings_df['domain_name']))
entity_to_domain_atc = dict(zip(atc_overrides_df['entity_name'], atc_overrides_df['domain_name']))

print(f"\nLoaded entity_to_concept: {len(entity_to_concept)} entities")
print(f"Loaded concept_to_domain: {len(concept_to_domain)} concepts")
print(f"Loaded entity_to_domain_atc (ATC-grounded overrides): {len(entity_to_domain_atc)} entities")

# ---- Build a quick lookup of all known entity names for the extraction step later ----
known_entities = set(G_E.nodes())
print(f"\nTotal known entities in G_E (for extraction/matching in later steps): {len(known_entities)}")

Loaded G_E: 3920 nodes, 103551 edges
Loaded G_C: 39 nodes, 339 edges
Loaded G_D: 16 nodes, 119 edges

Loaded entity_to_concept: 149 entities
Loaded concept_to_domain: 39 concepts
Loaded entity_to_domain_atc (ATC-grounded overrides): 74 entities

Total known entities in G_E (for extraction/matching in later steps): 3920


In [4]:
def check_entity_exists(entity_name, G_E):
    """
    Check whether an entity is present in the knowledge graph at all.
    Absence means "no knowledge," not "false" - this is the basis for
    the UNKNOWN decision state added in this component.
    """
    return entity_name in G_E.nodes()


def determine_entity_exists(subject, obj, G_E):
    """
    Both subject and object must be present in G_E for verification to
    proceed meaningfully. If either is missing, the fact is out of the
    knowledge graph's coverage entirely (UNKNOWN), not evaluated as
    VERIFIED or HALLUCINATION.
    """
    return check_entity_exists(subject, G_E) and check_entity_exists(obj, G_E)


# Quick sanity test
test_known = "Acetylsalicylic acid"
test_unknown = "Pancreatic disease"

print(f"'{test_known}' known: {check_entity_exists(test_known, G_E)}")
print(f"'{test_unknown}' known: {check_entity_exists(test_unknown, G_E)}")
print(f"Pair exists check (known + unknown): {determine_entity_exists(test_known, test_unknown, G_E)}")

'Acetylsalicylic acid' known: True
'Pancreatic disease' known: False
Pair exists check (known + unknown): False


In [5]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "gemma3:latest"

# ============================================================================
# Stage 2 (LLM-based, with few-shot examples): Question -> Candidate Triple
# ============================================================================

EXTRACTION_PROMPT_TEMPLATE = """You are extracting a structured fact from a medical question.

Given a question, identify:
1. subject: the drug being asked about (use its formal/generic pharmaceutical name, not brand name)
2. predicate: exactly one of [treats, causes, interacts_with] - pick the closest match
3. object: the condition, side effect, or second drug being asked about

Respond with ONLY a JSON object, nothing else - no explanation, no markdown formatting.

Examples:

Question: Will Aspirin be used for pancreatic disease?
{{"subject": "Acetylsalicylic acid", "predicate": "treats", "object": "Pancreatic disease"}}

Question: Does Ibuprofen cause stomach bleeding?
{{"subject": "Ibuprofen", "predicate": "causes", "object": "Stomach bleeding"}}

Question: Can I take Warfarin with Aspirin together?
{{"subject": "Warfarin", "predicate": "interacts_with", "object": "Acetylsalicylic acid"}}

Question: Is Metformin used to treat type 2 diabetes?
{{"subject": "Metformin", "predicate": "treats", "object": "Type 2 diabetes"}}

Question: Does Ranitidine cause headaches?
{{"subject": "Ranitidine", "predicate": "causes", "object": "Headache"}}

Now extract from this question:

Question: {question}
"""


def extract_triple_via_llm(question, model=OLLAMA_MODEL):
    """
    Call local Ollama Gemma3 (few-shot prompted) to extract a candidate
    (subject, predicate, object) triple from a free-text question.
    """
    prompt = EXTRACTION_PROMPT_TEMPLATE.format(question=question)

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.0}
            },
            timeout=60
        )
        response.raise_for_status()
        raw_output = response.json()["response"].strip()

        if raw_output.startswith("```"):
            raw_output = raw_output.split("```")[1]
            if raw_output.startswith("json"):
                raw_output = raw_output[4:]
        raw_output = raw_output.strip()

        parsed = json.loads(raw_output)
        return {
            'subject': parsed.get('subject'),
            'predicate': parsed.get('predicate'),
            'object': parsed.get('object'),
            'raw_llm_output': raw_output,
            'extraction_error': None,
        }
    except Exception as e:
        return {
            'subject': None, 'predicate': None, 'object': None,
            'raw_llm_output': None,
            'extraction_error': str(e),
        }


# Test on your example question
test_question = "Will Aspirin be used for pancreatic disease?"
llm_result = extract_triple_via_llm(test_question)
print(llm_result)

{'subject': 'Acetylsalicylic acid', 'predicate': 'treats', 'object': 'Pancreatic disease', 'raw_llm_output': '{"subject": "Acetylsalicylic acid", "predicate": "treats", "object": "Pancreatic disease"}', 'extraction_error': None}


In [6]:
# Test with a question NOT in the few-shot examples, to confirm genuine generalization
test_question_2 = "Can Diflunisal be taken alongside Levofloxacin safely?"
llm_result_2 = extract_triple_via_llm(test_question_2)
print(llm_result_2)

# And one more, testing a 'causes' relationship not seen verbatim
test_question_3 = "Does Phenylbutazone lead to bleeding problems?"
llm_result_3 = extract_triple_via_llm(test_question_3)
print(llm_result_3)

{'subject': 'Diflunisal', 'predicate': 'interacts_with', 'object': 'Levofloxacin', 'raw_llm_output': '{"subject": "Diflunisal", "predicate": "interacts_with", "object": "Levofloxacin"}', 'extraction_error': None}
{'subject': 'Phenylbutazone', 'predicate': 'causes', 'object': 'Bleeding problems', 'raw_llm_output': '{"subject": "Phenylbutazone", "predicate": "causes", "object": "Bleeding problems"}', 'extraction_error': None}


In [8]:
# ============================================================================
# Verified core functions, copied unchanged from Component 5
# (kept identical here for Component 6's standalone reproducibility -
#  these are NOT modified in any way from the validated version)
# ============================================================================

def verify_entity_level(subject, predicate, obj, G_E):
    if subject not in G_E.nodes() or obj not in G_E.nodes():
        return 0.0
    if G_E.has_edge(subject, obj):
        edge_data = G_E[subject][obj]
        if edge_data.get('predicate') == predicate:
            return 1.0
        else:
            return 0.3
    if predicate == 'interacts_with' and G_E.has_edge(obj, subject):
        edge_data = G_E[obj][subject]
        if edge_data.get('predicate') == predicate:
            return 1.0
    return 0.1


def verify_concept_level(subject, predicate, obj, G_C, entity_to_concept):
    subject_concept = entity_to_concept.get(subject)
    obj_concept = entity_to_concept.get(obj)
    if not subject_concept or not obj_concept:
        return 0.0
    if subject_concept not in G_C.nodes() or obj_concept not in G_C.nodes():
        return 0.0
    if G_C.has_edge(subject_concept, obj_concept):
        edge_data = G_C[subject_concept][obj_concept]
        if edge_data.get('predicate') == predicate:
            return 1.0
        else:
            return 0.4
    if predicate == 'interacts_with' and G_C.has_edge(obj_concept, subject_concept):
        edge_data = G_C[obj_concept][subject_concept]
        if edge_data.get('predicate') == predicate:
            return 1.0
    return 0.2


def verify_domain_level(subject, predicate, obj, G_D, entity_to_concept, concept_to_domain,
                        entity_to_domain_atc=None):
    if entity_to_domain_atc:
        subject_domain = entity_to_domain_atc.get(subject)
        obj_domain = entity_to_domain_atc.get(obj)
    else:
        subject_domain = None
        obj_domain = None

    if subject_domain is None:
        subject_concept = entity_to_concept.get(subject)
        subject_domain = concept_to_domain.get(subject_concept) if subject_concept else None

    if obj_domain is None:
        obj_concept = entity_to_concept.get(obj)
        obj_domain = concept_to_domain.get(obj_concept) if obj_concept else None

    if not subject_domain or not obj_domain:
        return 0.0

    if subject_domain not in G_D.nodes() or obj_domain not in G_D.nodes():
        return 0.0

    def _predicates_between(u, v):
        if not G_D.has_edge(u, v):
            return set()
        return {data.get('predicate') for _, data in G_D.get_edge_data(u, v).items()}

    forward_predicates = _predicates_between(subject_domain, obj_domain)

    if forward_predicates:
        if predicate in forward_predicates:
            return 1.0
        else:
            return 0.5

    if predicate == 'interacts_with':
        reverse_predicates = _predicates_between(obj_domain, subject_domain)
        if predicate in reverse_predicates:
            return 1.0

    return 0.3


def calculate_cross_scale_consistency(S_E, S_C, S_D):
    c_EC = 1.0 - abs(S_E - S_C)
    c_CD = 1.0 - abs(S_C - S_D)
    c_ED = 1.0 - abs(S_E - S_D)
    C_f = (c_EC * c_CD * c_ED) ** (1/3)
    return {'c_EC': c_EC, 'c_CD': c_CD, 'c_ED': c_ED, 'C_f': C_f}


def detect_inconsistency_patterns(S_E, S_C, S_D,
                                  tau_high=0.7, tau_low=0.3, delta_threshold=0.5):
    I1 = (S_E > tau_high) and (S_D < tau_low)
    I2 = (S_C > tau_high) and (S_D < tau_low)
    divergences = [abs(S_E - S_C), abs(S_C - S_D), abs(S_E - S_D)]
    max_divergence = max(divergences)
    I3 = max_divergence > delta_threshold

    patterns_detected = []
    if I1:
        patterns_detected.append('I1_Entity_Domain_Contradiction')
    if I2:
        patterns_detected.append('I2_Concept_Domain_Contradiction')
    if I3:
        patterns_detected.append('I3_Scale_Divergence')

    return {'I1': I1, 'I2': I2, 'I3': I3, 'max_divergence': max_divergence,
            'patterns_detected': patterns_detected, 'num_patterns': len(patterns_detected)}


def calculate_confidence(S_E, S_C, S_D, C_f,
                         alpha=0.3, beta=0.25, gamma=0.2, delta=0.25):
    return alpha * S_E + beta * S_C + gamma * S_D + delta * C_f


def detect_hallucination(S_E, S_C, S_D,
                         theta_consistency=0.5, theta_confidence=0.55,
                         tau_high=0.7, tau_low=0.3, delta_threshold=0.5,
                         alpha=0.3, beta=0.25, gamma=0.2, delta=0.25,
                         entity_exists=True):
    if not entity_exists:
        return {
            'decision': 'UNKNOWN',
            'reason': 'Entity not present in knowledge graph — cannot verify',
            'level_triggered': 0,
            'scores': {'S_E': S_E, 'S_C': S_C, 'S_D': S_D, 'C_f': None, 'Conf_f': None},
            'patterns': {'patterns_detected': []},
        }

    consistency = calculate_cross_scale_consistency(S_E, S_C, S_D)
    C_f = consistency['C_f']
    patterns = detect_inconsistency_patterns(S_E, S_C, S_D, tau_high, tau_low, delta_threshold)
    Conf_f = calculate_confidence(S_E, S_C, S_D, C_f, alpha, beta, gamma, delta)

    if C_f < theta_consistency:
        decision, reason, level = 'HALLUCINATION', f'Low cross-scale consistency (C(f)={C_f:.3f} < {theta_consistency})', 1
    elif patterns['num_patterns'] > 0:
        decision, reason, level = 'HALLUCINATION', f"Scale inconsistency patterns: {patterns['patterns_detected']}", 2
    elif Conf_f < theta_confidence:
        decision, reason, level = 'HALLUCINATION', f'Low overall confidence (Conf(f)={Conf_f:.3f} < {theta_confidence})', 3
    else:
        decision, reason, level = 'VERIFIED', f'All checks passed (C(f)={C_f:.3f}, Conf(f)={Conf_f:.3f})', None

    return {
        'decision': decision, 'reason': reason, 'level_triggered': level,
        'scores': {'S_E': S_E, 'S_C': S_C, 'S_D': S_D, 'C_f': C_f, 'Conf_f': Conf_f},
        'patterns': patterns,
    }

In [9]:
# ============================================================================
# Stage 3: Entity Existence Check + Call the Validated Verifier
# ============================================================================
# This is where LLM-proposed entities meet the deterministic graph.
# The LLM's extraction is only a PROPOSAL - nothing here trusts it as fact.
# check_entity_exists / determine_entity_exists (from Cell 2) do the real work.

def run_pipeline_on_question(question, G_E, G_C, G_D,
                             entity_to_concept, concept_to_domain, entity_to_domain_atc):
    """
    Full Stage 1-3 flow: LLM extracts a candidate triple, we check entity
    existence, and if both entities are known we call the unchanged,
    validated multi-scale verifier. If either entity is unknown, we report
    UNKNOWN rather than attempting to verify - and never fabricate a triple
    the extractor couldn't confidently produce.
    """
    # Stage 1: LLM extraction
    extracted = extract_triple_via_llm(question)

    if extracted['extraction_error']:
        return {
            'question': question,
            'decision': 'EXTRACTION_FAILED',
            'reason': f"LLM extraction error: {extracted['extraction_error']}",
            'extracted': extracted,
        }

    subject = extracted['subject']
    predicate = extracted['predicate']
    obj = extracted['object']

    # Guard: if extraction produced incomplete triple, cannot proceed
    if not subject or not predicate or not obj:
        return {
            'question': question,
            'decision': 'EXTRACTION_INCOMPLETE',
            'reason': f"Could not extract a complete triple (subject={subject}, "
                      f"predicate={predicate}, object={obj})",
            'extracted': extracted,
        }

    # Guard: predicate must be one of the three known types
    if predicate not in ('treats', 'causes', 'interacts_with'):
        return {
            'question': question,
            'decision': 'EXTRACTION_INCOMPLETE',
            'reason': f"Extracted predicate '{predicate}' is not one of the "
                      f"three known relationship types",
            'extracted': extracted,
        }

    # Stage 2: entity existence check (deterministic, not LLM-trusted)
    subject_known = check_entity_exists(subject, G_E)
    object_known = check_entity_exists(obj, G_E)

    if not (subject_known and object_known):
        missing = []
        if not subject_known:
            missing.append(f"subject '{subject}'")
        if not object_known:
            missing.append(f"object '{obj}'")
        return {
            'question': question,
            'decision': 'UNKNOWN',
            'reason': f"Not in knowledge graph coverage: {', '.join(missing)}",
            'extracted': extracted,
            'subject_known': subject_known,
            'object_known': object_known,
        }

    # Stage 3: both entities known - run the actual, unchanged, validated verifier
    S_E = verify_entity_level(subject, predicate, obj, G_E)
    S_C = verify_concept_level(subject, predicate, obj, G_C, entity_to_concept)
    S_D = verify_domain_level(subject, predicate, obj, G_D, entity_to_concept,
                              concept_to_domain, entity_to_domain_atc)

    result = detect_hallucination(S_E, S_C, S_D, entity_exists=True)

    return {
        'question': question,
        'decision': result['decision'],
        'reason': result['reason'],
        'extracted': extracted,
        'triple': (subject, predicate, obj),
        'scores': result['scores'],
    }


# Test on all three questions so far
for q in [
    "Will Aspirin be used for pancreatic disease?",
    "Can Diflunisal be taken alongside Levofloxacin safely?",
    "Does Phenylbutazone lead to bleeding problems?",
]:
    result = run_pipeline_on_question(q, G_E, G_C, G_D,
                                      entity_to_concept, concept_to_domain, entity_to_domain_atc)
    print(f"\nQ: {result['question']}")
    print(f"Decision: {result['decision']}")
    print(f"Reason: {result['reason']}")
    if 'triple' in result:
        print(f"Triple: {result['triple']}")


Q: Will Aspirin be used for pancreatic disease?
Decision: UNKNOWN
Reason: Not in knowledge graph coverage: object 'Pancreatic disease'

Q: Can Diflunisal be taken alongside Levofloxacin safely?
Decision: VERIFIED
Reason: All checks passed (C(f)=1.000, Conf(f)=1.000)
Triple: ('Diflunisal', 'interacts_with', 'Levofloxacin')

Q: Does Phenylbutazone lead to bleeding problems?
Decision: UNKNOWN
Reason: Not in knowledge graph coverage: object 'Bleeding problems'


In [11]:
# ============================================================================
# Stage 4 (v2): Generate ~1000 natural, patient-style questions from
# test_facts_large_v2.csv, preserving full ground-truth traceability.
# ============================================================================

import pandas as pd
import requests
import json
import time

test_facts_df = pd.read_csv(INTERMEDIATE_DIR / "test_facts_large_v2.csv")
print(f"Source facts available: {len(test_facts_df)}")
print(test_facts_df.groupby(['category', 'ground_truth']).size())

QUESTION_GEN_PROMPT_TEMPLATE = """You are simulating how a real patient would ask their doctor a question,
based on a medical fact. Do NOT state whether the fact is true or false -
just phrase it as a natural, everyday question a patient might genuinely ask.

Vary your phrasing naturally - patients ask things like "will this help with...",
"is it safe if I...", "could this cause...", "should I be worried about...",
"my doctor gave me X, will it work for...", etc. Keep it to one or two sentences,
casual and human, not clinical or textbook-sounding.

Respond with ONLY the question text, nothing else - no quotes, no explanation.

Examples:

Fact: (Ibuprofen, treats, Pain)
Question: Will ibuprofen actually help with my pain, or should I take something stronger?

Fact: (Warfarin, interacts_with, Aspirin)
Question: I'm on warfarin already, is it risky to also take an aspirin for my headache?

Fact: (Metformin, causes, Nausea)
Question: I just started metformin and I've been feeling nauseous, could that be from the medication?

Fact: ({subject}, {predicate}, {object})
Question:"""


def generate_natural_question(subject, predicate, obj, model=OLLAMA_MODEL, retries=2):
    """
    Use Gemma3 to paraphrase a (subject, predicate, object) triple into a
    natural, patient-style question. Ground truth (the original triple)
    is preserved separately - the generated question is never treated as
    a new source of truth.
    """
    prompt = QUESTION_GEN_PROMPT_TEMPLATE.format(subject=subject, predicate=predicate, object=obj)

    for attempt in range(retries + 1):
        try:
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": model,
                    "prompt": prompt,
                    "stream": False,
                    "options": {"temperature": 0.7}  # some variety in phrasing is desired here
                },
                timeout=60
            )
            response.raise_for_status()
            question = response.json()["response"].strip()
            # strip stray quotes some models add
            question = question.strip('"').strip()
            if question:
                return question, None
        except Exception as e:
            if attempt == retries:
                return None, str(e)
            time.sleep(1)
    return None, "Unknown failure"


# ---- Quick test on 3 rows before committing to the full 974 ----
test_sample = test_facts_df.sample(3, random_state=1)
print("\n--- Quick test on 3 sample facts ---")
for _, row in test_sample.iterrows():
    q, err = generate_natural_question(row['subject'], row['predicate'], row['object'])
    print(f"\nFact: ({row['subject']}, {row['predicate']}, {row['object']})  [{row['category']}]")
    print(f"Generated question: {q}")
    if err:
        print(f"Error: {err}")

Source facts available: 974
category                  ground_truth 
domain_violation          HALLUCINATION     75
nonexistent_relationship  HALLUCINATION    299
real_edge                 VERIFIED         400
wrong_predicate           HALLUCINATION    200
dtype: int64

--- Quick test on 3 sample facts ---

Fact: (Cyclobenzaprine, interacts_with, Gilteritinib)  [real_edge]
Generated question: My doctor prescribed cyclobenzaprine, will that interact with my gilteritinib?

Fact: (Acepromazine, treats, Ganciclovir)  [nonexistent_relationship]
Generated question: My doctor prescribed me acepromazine, will it actually help with the ganciclovir?

Fact: (Hydrocodone, interacts_with, Romidepsin)  [real_edge]
Generated question: My doctor prescribed me hydrocodone, will that interact with my other medication, romidepsin?


In [12]:
# ============================================================================
# Stage 4 (v2 continued): Mid-size batch check (n=50) before full run
# ============================================================================

import time

batch_sample = test_facts_df.sample(50, random_state=2).reset_index(drop=True)
batch_results = []

start = time.time()
for i, row in batch_sample.iterrows():
    q, err = generate_natural_question(row['subject'], row['predicate'], row['object'])
    batch_results.append({
        'subject': row['subject'],
        'predicate': row['predicate'],
        'object': row['object'],
        'category': row['category'],
        'ground_truth': row['ground_truth'],
        'generated_question': q,
        'generation_error': err,
    })
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        print(f"  {i+1}/50 done ({elapsed:.1f}s elapsed, ~{elapsed/(i+1):.2f}s/question)")

batch_df = pd.DataFrame(batch_results)

print(f"\nTotal time: {time.time() - start:.1f}s")
print(f"Failures: {batch_df['generation_error'].notna().sum()}/50")
print(f"\n--- Sample of 10 generated questions ---")
for _, row in batch_df.sample(10, random_state=3).iterrows():
    print(f"[{row['category']:25s}] {row['generated_question']}")

  10/50 done (10.3s elapsed, ~1.03s/question)
  20/50 done (21.1s elapsed, ~1.05s/question)
  30/50 done (31.3s elapsed, ~1.04s/question)
  40/50 done (42.4s elapsed, ~1.06s/question)
  50/50 done (53.7s elapsed, ~1.07s/question)

Total time: 53.8s
Failures: 0/50

--- Sample of 10 generated questions ---
[real_edge                ] My doctor prescribed ganciclovir, will it interact with topotecan if I’m ever taking that?
[nonexistent_relationship ] I’ve heard this treatment is supposed to help with diabetes, but will it actually work long-term?
[real_edge                ] My doctor prescribed me phenylbutazone, will it interact with promethazine that I take for my allergies?
[nonexistent_relationship ] My doctor prescribed cariprazine, will it interact with Erdafitinib?
[wrong_predicate          ] My doctor prescribed methotrimeprazine for my anxiety, will it interact with any of my other medications, like my schizophrenia medication?
[wrong_predicate          ] My doctor prescribed si

In [13]:
# ============================================================================
# Stage 4 (v2 final): Full run - generate natural questions for all 974 facts
# Checkpointed every 100 questions in case of interruption.
# ============================================================================

import time

CHECKPOINT_PATH = INTERMEDIATE_DIR / "component6_natural_questions_checkpoint.csv"
FINAL_OUTPUT_PATH = INTERMEDIATE_DIR / "component6_natural_questions.csv"
CHECKPOINT_EVERY = 100

all_facts = test_facts_df.reset_index(drop=True)
results = []

start = time.time()
print(f"Generating natural questions for {len(all_facts)} facts...")

for i, row in all_facts.iterrows():
    q, err = generate_natural_question(row['subject'], row['predicate'], row['object'])
    results.append({
        'subject': row['subject'],
        'predicate': row['predicate'],
        'object': row['object'],
        'category': row['category'],
        'ground_truth': row['ground_truth'],
        'generated_question': q,
        'generation_error': err,
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - start
        remaining = (elapsed / (i + 1)) * (len(all_facts) - (i + 1))
        print(f"  {i+1}/{len(all_facts)} done "
              f"({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")
        pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)

final_df = pd.DataFrame(results)
final_df.to_csv(FINAL_OUTPUT_PATH, index=False)

total_time = time.time() - start
print(f"\nDone. Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Failures: {final_df['generation_error'].notna().sum()}/{len(final_df)}")
print(f"\nComposition:")
print(final_df.groupby(['category', 'ground_truth']).size())
print(f"\nSaved: {FINAL_OUTPUT_PATH}")

Generating natural questions for 974 facts...
  100/974 done (104s elapsed, ~908s remaining)
  200/974 done (207s elapsed, ~803s remaining)
  300/974 done (310s elapsed, ~696s remaining)
  400/974 done (414s elapsed, ~593s remaining)
  500/974 done (516s elapsed, ~489s remaining)
  600/974 done (619s elapsed, ~386s remaining)
  700/974 done (722s elapsed, ~283s remaining)
  800/974 done (825s elapsed, ~180s remaining)
  900/974 done (929s elapsed, ~76s remaining)

Done. Total time: 1005s (16.7 min)
Failures: 0/974

Composition:
category                  ground_truth 
domain_violation          HALLUCINATION     75
nonexistent_relationship  HALLUCINATION    299
real_edge                 VERIFIED         400
wrong_predicate           HALLUCINATION    200
dtype: int64

Saved: ../Datasets/intermediate/component6_natural_questions.csv


In [14]:
# ============================================================================
# Stage 5: Full end-to-end run - natural questions -> extraction ->
# verification -> decision, scored against preserved ground truth.
# Checkpointed every 100 for safety.
# ============================================================================

import time

natural_questions_df = pd.read_csv(INTERMEDIATE_DIR / "component6_natural_questions.csv")
print(f"Loaded {len(natural_questions_df)} natural questions")

CHECKPOINT_PATH_E2E = INTERMEDIATE_DIR / "component6_e2e_results_checkpoint.csv"
FINAL_OUTPUT_PATH_E2E = INTERMEDIATE_DIR / "component6_e2e_results.csv"
CHECKPOINT_EVERY = 100

e2e_results = []
start = time.time()

for i, row in natural_questions_df.iterrows():
    question = row['generated_question']

    pipeline_result = run_pipeline_on_question(
        question, G_E, G_C, G_D, entity_to_concept, concept_to_domain, entity_to_domain_atc
    )

    e2e_results.append({
        'original_subject': row['subject'],
        'original_predicate': row['predicate'],
        'original_object': row['object'],
        'category': row['category'],
        'ground_truth': row['ground_truth'],
        'generated_question': question,
        'pipeline_decision': pipeline_result['decision'],
        'pipeline_reason': pipeline_result.get('reason'),
        'extracted_subject': pipeline_result.get('extracted', {}).get('subject'),
        'extracted_predicate': pipeline_result.get('extracted', {}).get('predicate'),
        'extracted_object': pipeline_result.get('extracted', {}).get('object'),
    })

    if (i + 1) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - start
        remaining = (elapsed / (i + 1)) * (len(natural_questions_df) - (i + 1))
        print(f"  {i+1}/{len(natural_questions_df)} done "
              f"({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")
        pd.DataFrame(e2e_results).to_csv(CHECKPOINT_PATH_E2E, index=False)

e2e_df = pd.DataFrame(e2e_results)
e2e_df.to_csv(FINAL_OUTPUT_PATH_E2E, index=False)

total_time = time.time() - start
print(f"\nDone. Total time: {total_time:.0f}s ({total_time/60:.1f} min)")

# ============================================================================
# Three-way scoring: resolution rate, UNKNOWN rate, accuracy-on-resolved
# ============================================================================

total = len(e2e_df)
unknown_count = (e2e_df['pipeline_decision'] == 'UNKNOWN').sum()
extraction_failed_count = e2e_df['pipeline_decision'].isin(['EXTRACTION_FAILED', 'EXTRACTION_INCOMPLETE']).sum()
resolved_df = e2e_df[e2e_df['pipeline_decision'].isin(['VERIFIED', 'HALLUCINATION'])]

print(f"\n--- THREE-WAY BREAKDOWN ---")
print(f"Total questions: {total}")
print(f"Resolved (VERIFIED/HALLUCINATION decision reached): {len(resolved_df)} ({len(resolved_df)/total*100:.1f}%)")
print(f"UNKNOWN (entity not found in graph): {unknown_count} ({unknown_count/total*100:.1f}%)")
print(f"Extraction failed/incomplete: {extraction_failed_count} ({extraction_failed_count/total*100:.1f}%)")

if len(resolved_df) > 0:
    correct = (resolved_df['pipeline_decision'] == resolved_df['ground_truth']).sum()
    accuracy_on_resolved = correct / len(resolved_df)
    print(f"\nAccuracy on resolved subset: {correct}/{len(resolved_df)} = {accuracy_on_resolved*100:.1f}%")

    print(f"\nAccuracy on resolved subset, by category:")
    for cat in resolved_df['category'].unique():
        cat_df = resolved_df[resolved_df['category'] == cat]
        cat_correct = (cat_df['pipeline_decision'] == cat_df['ground_truth']).sum()
        print(f"  {cat:28s} n={len(cat_df):4d}  accuracy={cat_correct/len(cat_df)*100:5.1f}%")

print(f"\nSaved: {FINAL_OUTPUT_PATH_E2E}")

Loaded 974 natural questions
  100/974 done (130s elapsed, ~1137s remaining)
  200/974 done (259s elapsed, ~1003s remaining)
  300/974 done (387s elapsed, ~869s remaining)
  400/974 done (514s elapsed, ~738s remaining)
  500/974 done (641s elapsed, ~608s remaining)
  600/974 done (768s elapsed, ~478s remaining)
  700/974 done (897s elapsed, ~351s remaining)
  800/974 done (1973s elapsed, ~429s remaining)
  900/974 done (2276s elapsed, ~187s remaining)

Done. Total time: 2575s (42.9 min)

--- THREE-WAY BREAKDOWN ---
Total questions: 974
Resolved (VERIFIED/HALLUCINATION decision reached): 733 (75.3%)
UNKNOWN (entity not found in graph): 238 (24.4%)
Extraction failed/incomplete: 3 (0.3%)

Accuracy on resolved subset: 555/733 = 75.7%

Accuracy on resolved subset, by category:
  nonexistent_relationship     n= 170  accuracy= 98.2%
  real_edge                    n= 359  accuracy= 77.4%
  wrong_predicate              n= 154  accuracy= 39.0%
  domain_violation             n=  50  accuracy=100.

In [15]:
import pandas as pd
e2e_df = pd.read_csv("../Datasets/intermediate/component6_e2e_results.csv")

wp_resolved = e2e_df[
    (e2e_df['category'] == 'wrong_predicate') &
    (e2e_df['pipeline_decision'].isin(['VERIFIED', 'HALLUCINATION']))
    ]

wp_wrong = wp_resolved[wp_resolved['pipeline_decision'] != wp_resolved['ground_truth']]

print(f"wrong_predicate errors: {len(wp_wrong)}/{len(wp_resolved)}")
print(f"\nSample: original predicate vs extracted predicate")
print(wp_wrong[['original_predicate', 'extracted_predicate', 'generated_question', 'pipeline_decision']].head(15).to_string(index=False))

print(f"\nHow often does extracted_predicate differ from original_predicate?")
print((wp_wrong['original_predicate'] != wp_wrong['extracted_predicate']).sum(), "/", len(wp_wrong))

wrong_predicate errors: 94/154

Sample: original predicate vs extracted predicate
original_predicate extracted_predicate                                                                                              generated_question pipeline_decision
            treats      interacts_with My doctor prescribed diflunisal for my arthritis, will it interact with methotrimeprazine that I'm also taking?          VERIFIED
            treats      interacts_with                        My doctor prescribed lomefloxacin, will it interact with oxaprozin that I'm also taking?          VERIFIED
            treats      interacts_with                                   My doctor prescribed ifosfamide, will it work in conjunction with everolimus?          VERIFIED
            treats      interacts_with                      I was prescribed Famotidine with my Cefuroxime, will that be safe for me to take together?          VERIFIED
            treats      interacts_with                      My doctor pre

In [16]:
# ============================================================================
# Component 6: Full metric suite on the resolved subset, with the same
# structure as Component 5's evaluation, for direct comparability.
# ============================================================================

import pandas as pd

e2e_df = pd.read_csv(INTERMEDIATE_DIR / "component6_e2e_results.csv")

resolved = e2e_df[e2e_df['pipeline_decision'].isin(['VERIFIED', 'HALLUCINATION'])].copy()

def compute_full_metrics(df, decision_col='pipeline_decision', truth_col='ground_truth'):
    TP = len(df[(df[truth_col] == 'HALLUCINATION') & (df[decision_col] == 'HALLUCINATION')])
    TN = len(df[(df[truth_col] == 'VERIFIED') & (df[decision_col] == 'VERIFIED')])
    FP = len(df[(df[truth_col] == 'VERIFIED') & (df[decision_col] == 'HALLUCINATION')])
    FN = len(df[(df[truth_col] == 'HALLUCINATION') & (df[decision_col] == 'VERIFIED')])

    total = len(df)
    accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
            'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}


# ---- Overall, on resolved subset ----
overall = compute_full_metrics(resolved)
print("=" * 70)
print("COMPONENT 6 — FULL METRICS (resolved subset, n={})".format(len(resolved)))
print("=" * 70)
print(f"Accuracy:  {overall['accuracy']*100:.1f}%")
print(f"Precision: {overall['precision']*100:.1f}%")
print(f"Recall:    {overall['recall']*100:.1f}%")
print(f"F1-Score:  {overall['f1']*100:.1f}%")
print(f"TP={overall['TP']}  TN={overall['TN']}  FP={overall['FP']}  FN={overall['FN']}")

# ---- Per-category ----
print("\n" + "=" * 70)
print("PER-CATEGORY METRICS")
print("=" * 70)
category_metrics = {}
for cat in resolved['category'].unique():
    cat_df = resolved[resolved['category'] == cat]
    m = compute_full_metrics(cat_df)
    category_metrics[cat] = m
    print(f"\n{cat} (n={len(cat_df)}):")
    print(f"  Accuracy: {m['accuracy']*100:.1f}%  Precision: {m['precision']*100:.1f}%  "
          f"Recall: {m['recall']*100:.1f}%  F1: {m['f1']*100:.1f}%")

# ---- Coverage-adjusted metrics: what if UNKNOWN + extraction-failed count as errors? ----
# This is the CONSERVATIVE / worst-case framing - treating any non-answer as a failure.
# Reported separately, not blended into the primary metric, since it conflates two
# different failure modes (wrong answer vs. no answer).
print("\n" + "=" * 70)
print("COVERAGE CONTEXT (reported separately, not blended into primary metrics)")
print("=" * 70)
total_all = len(e2e_df)
resolved_n = len(resolved)
unknown_n = (e2e_df['pipeline_decision'] == 'UNKNOWN').sum()
failed_n = e2e_df['pipeline_decision'].isin(['EXTRACTION_FAILED', 'EXTRACTION_INCOMPLETE']).sum()

print(f"Total questions: {total_all}")
print(f"  Resolved (metrics computed above): {resolved_n} ({resolved_n/total_all*100:.1f}%)")
print(f"  UNKNOWN (correctly deferred, not wrong): {unknown_n} ({unknown_n/total_all*100:.1f}%)")
print(f"  Extraction failed: {failed_n} ({failed_n/total_all*100:.1f}%)")

# Worst-case: treat UNKNOWN/failed as if they were errors (lower bound framing)
worst_case_correct = overall['TP'] + overall['TN']
worst_case_accuracy = worst_case_correct / total_all
print(f"\nWorst-case accuracy (if all non-resolved were counted as errors): "
      f"{worst_case_correct}/{total_all} = {worst_case_accuracy*100:.1f}%")
print("(This is a conservative lower bound, not recommended as the primary reported")
print(" metric, since it treats a deliberate safe non-answer the same as a wrong answer.)")

# Save
import json
output = {
    'overall_metrics_resolved_subset': overall,
    'category_metrics_resolved_subset': category_metrics,
    'coverage': {
        'total': int(total_all),
        'resolved': int(resolved_n),
        'unknown': int(unknown_n),
        'extraction_failed': int(failed_n),
    },
    'worst_case_accuracy_all_facts': float(worst_case_accuracy),
}
with open(INTERMEDIATE_DIR / "component6_full_metrics.json", 'w') as f:
    json.dump(output, f, indent=2)
print(f"\nSaved: {INTERMEDIATE_DIR / 'component6_full_metrics.json'}")

COMPONENT 6 — FULL METRICS (resolved subset, n=733)
Accuracy:  75.7%
Precision: 77.4%
Recall:    74.1%
F1-Score:  75.7%
TP=277  TN=278  FP=81  FN=97

PER-CATEGORY METRICS

nonexistent_relationship (n=170):
  Accuracy: 98.2%  Precision: 100.0%  Recall: 98.2%  F1: 99.1%

real_edge (n=359):
  Accuracy: 77.4%  Precision: 0.0%  Recall: 0.0%  F1: 0.0%

wrong_predicate (n=154):
  Accuracy: 39.0%  Precision: 100.0%  Recall: 39.0%  F1: 56.1%

domain_violation (n=50):
  Accuracy: 100.0%  Precision: 100.0%  Recall: 100.0%  F1: 100.0%

COVERAGE CONTEXT (reported separately, not blended into primary metrics)
Total questions: 974
  Resolved (metrics computed above): 733 (75.3%)
  UNKNOWN (correctly deferred, not wrong): 238 (24.4%)
  Extraction failed: 3 (0.3%)

Worst-case accuracy (if all non-resolved were counted as errors): 555/974 = 57.0%
(This is a conservative lower bound, not recommended as the primary reported
 metric, since it treats a deliberate safe non-answer the same as a wrong answer.)